<a class="anchor cute_anchor" id="Import_0"></a>
# <span class="cute_title" style="color: #2a9466">Import</span>

In [3]:
# Standard library imports
import os
import sys
from datetime import date

import re


# Local application imports
sys.path.append('..')
sys.path.append('../..')

from file_management import get_files_dir,check_save_file
from text_analysis import remove_stopwords,remove_adj_adv

# Get file directories
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

# Third-party imports
import pandas as pd
import numpy as np



In [5]:
from retrieve_organism import *

# Input and Output files

## Input

Articles that fit into the metabolic engineering classification

In [8]:
# Classify articles with full text
# file = OUTPUT_DIR + '/Articles/classified_articles_v_2025_06_30.csv'
file_path = OUTPUT_DIR / 'full_text_articles.json.gz'

# Define memory-efficient dtype mapping
dtype_spec = {
    'PM_ID': 'int32',      # PubMed IDs can be large, but int32 is sufficient for most cases
    'Year': 'int16',       # Years (2000-2025) easily fit in int16
    'PMC_ID': 'Int32',      # Use nullable integer type for PMC IDs (may contain missing values)
    'Title': 'string',
    'Abstract': 'string',
    'Journal': 'category', 
    'DOI': 'string',
    'Type': 'string',   
    'Author': 'string', 
    'Text': 'string'
}

# Load data with optimized memory usage
relevant_articles = pd.read_json(
    file_path,
    dtype=dtype_spec,
    orient='index',
    compression='gzip'
)

# Filter out review articles (case-insensitive check)
relevant_articles = relevant_articles.loc[
    ~relevant_articles['Type'].str.lower().str.contains('review', na=False)
]

relevant_articles.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",<NA>
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,<NA>,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",<NA>
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",<NA>
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,<NA>,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",<NA>


In [9]:
relevant_articles = relevant_articles.loc[~relevant_articles.Type.str.lower().str.contains('review'),]
relevant_articles = relevant_articles.loc[~relevant_articles.Title.str.lower().str.contains('review'),]
#relevant_articles = relevant_articles.loc[~relevant_articles.Abstract.str.lower().str.contains('review'),]

## Output

In [10]:
general_name = 'full_text_classified_w_org'

today = date.today()
today = today.strftime("%y_%m_%d")

output_file = f'{general_name}_{today}.json'
output_file

'full_text_classified_w_org_26_02_02.json'

## File processing

### Normalize organism names

In [11]:
synonym = {
    'Pichia pastoris':'Komagataella pastoris',
    'Komagataella phaffii':'Komagataella pastoris',
    'Ralstonia eutropha':'Cupriavidus necator',
    'Streptomyces roseosporus':'Streptomyces filamentosus',
    'Clostridium thermocellum':'Acetivibrio thermocellus',
    'Rhodobacter sphaeroides':'Cereibacter sphaeroides',
    'Bacillus megaterium':'Priestia megaterium',
    'Bacillus circulans':'Niallia circulans'
}

synonym.update({
    'Streptomyces tsukubaensis':'Streptomyces tsukubensis',
    'Alcaligenes eutrophus':'Cupriavidus necator',
    'Bacillus typhimurium':'Salmonella enterica',
    'Salmonella typhimurium':'Salmonella enterica',
    'Aeromonas punctata':'Aeromonas caviae',
    'Streptomyces aureofaciens':'Kitasatospora aureofaciens',
    'Pichia ciferrii':'Wickerhamomyces ciferrii',
    'Lactobacillus fermentum':'Limosilactobacillus fermentum',
    'Ashbya gossypii':'Eremothecium gossypii',
    'Candida peltata':'Nakazawaea peltata',
    'Hansenula polymorpha':'Ogataea polymorpha',
    'Candida famata':'Debaryomyces hansenii',
    'Torulopsis glabrata':'Nakaseomyces glabratus',
    'Candida glabrata':'Nakaseomyces glabratus',
    'Candida utilis':'Cyberlindnera jadinii',
    'Lactobacillus plantarum':'Lactiplantibacillus plantarum',
    'Lactobacillus rhamnosus':'Lacticaseibacillus rhamnosus',
    'Lactobacillus reuteri':'Limosilactobacillus reuteri',
    'Lactobacillus brevis':'Levilactobacillus brevis',
    'Lactobacillus diolivorans':'Lentilactobacillus diolivorans',
    'Lactobacillus panis':'Limosilactobacillus panis',
    'Lactobacillus pentosus':'Lactiplantibacillus pentosus',
    'Lactobacillus kefir':'Lentilactobacillus kefiri',
    'Lactobacillus buchneri':'Lentilactobacillus buchneri',
    'Lactobacillus paracasei':'Lacticaseibacillus paracasei',
    'Lactobacillus casei':'Lacticaseibacillus casei',
    #'PCC 6803':'PCC 6803',
    
    'Candida bombicola':'Starmerella bombicola',
    'Candida tenuis':'Yamadazyma tenuis',
    'Candida antarctica':'Moesziomyces antarcticus',
    'Candida lignohabitans':'Sugiyamaella lignohabitans',
    'Candida guilliermondii':'Meyerozyma guilliermondii',
    'Candida glycerinogenes':'Pichia kudriavzevii',
    'Xanthophyllomyces dendrorhous':'Phaffia rhodozyma',
    'Rhodosporidium toruloides':'Rhodotorula toruloides',
    'Clostridium cellulolyticum':'Ruminiclostridium cellulolyticum',    
    'Hungateiclostridium thermocellum':'Acetivibrio thermocellus',
    'Clostridium termitidis': 'Ruminiclostridium cellobioparum',
    
    'Mycobacterium neoaurum':'Mycolicibacterium neoaurum',
    'Mycobacterium smegmatis':'Mycolicibacterium smegmatis',
    'Mycobacterium fortuitum':'Mycolicibacterium fortuitum'
})


In [12]:
synonym.update({
    'Saccharamyces cerevisiae':'Saccharomyces cerevisiae',
    'Sacchromyces cerevisiae':'Saccharomyces cerevisiae',
    'Saccharomyees cerevisiae':'Saccharomyces cerevisiae',
    'Sachharomyces cerevisiae':'Saccharomyces cerevisiae',
})


In [13]:

# Map lowercase old-name -> new-name
syn_lc = {k.lower(): v for k, v in synonym.items()}

# One combined pattern, longest first helps when keys overlap
keys_sorted = sorted(synonym.keys(), key=len, reverse=True)

# \b ... \b prevents partial hits; re.escape avoids regex issues in names
pattern = re.compile(r"\b(" + "|".join(map(re.escape, keys_sorted)) + r")\b", flags=re.IGNORECASE)


In [14]:
cols = ["Text", "Abstract", "Title"]

def repl(match: re.Match) -> str:
    return syn_lc[match.group(0).lower()]

for col in cols:
    relevant_articles[col] = relevant_articles[col].str.replace(pattern, repl, regex=True)


### Normalize and format file text

Remove special characters and separate the words to find the organisms


In [15]:
# Remove stopwords
relevant_articles.loc[:,'Abstract_min'] = relevant_articles.loc[:,'Abstract'].map(remove_stopwords)
relevant_articles.loc[:,'Abstract_min'] = relevant_articles.loc[:,'Abstract_min'].map(remove_adj_adv)

relevant_articles.loc[:,'Title_min'] = relevant_articles.loc[:,'Title'].map(remove_stopwords)
relevant_articles.loc[:,'Title_min'] = relevant_articles.loc[:,'Title_min'].map(remove_adj_adv)

In [16]:
relevant_articles['Full_text_min'] = relevant_articles['Text'].map(lambda x: remove_stopwords(x) if pd.notna(x) else np.nan)
relevant_articles['Full_text_min'] = relevant_articles['Full_text_min'].map(lambda x: remove_adj_adv(x) if pd.notna(x) else np.nan)

In [17]:
#most common in title
words_to_remove = set(['production', 'engineered', 'engineering','pathway','gene',
                       'expression','using','produced','biosynthesis','cell','yield',
                   'synthesis','strains','carbon','produce','fermentation','via','product'])


In [18]:
# most common in abstract
words_to_remove.update(['growth', 'strain', 'increase', 'study','enzyme',
                   'activity','titer','protein','system','result','ethanol','strategy',
                   'compared','pathways','showed','could','flux','improve','analysis',
                   'metabolism','however','here','synthase','conversion','acid'])


In [19]:
#most common in full_text
words_to_remove.update(['glucose', 'figure', 'table', 'containing','plasmid','concentration',
                   'performed','rate','culture','dna','level','shown','promoter',
                   'control','reaction','pcr','condition','obtain','may','substrate',
                   'data','however','here','culture','respectively','sequence','result','metabolic'])
words_to_remove.update(['xylose','use','dehydrogenase','biomass','including','encoding','approach'])

words_to_remove = set(words_to_remove)
pattern = r'\b(' + '|'.join(words_to_remove) + r')e?s?d?i?n?g?\b'

# Remove words and clean up extra spaces
relevant_articles.loc[:, 'Abstract_min'] = (
    relevant_articles.loc[:, 'Abstract_min']
    .str.replace(pattern, '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with single space
    .str.strip()  # Remove leading/trailing spaces
)
# Remove words and clean up extra spaces
relevant_articles.loc[:, 'Title_min'] = (
    relevant_articles.loc[:, 'Title_min']
    .str.replace(pattern, '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with single space
    .str.strip()  # Remove leading/trailing spaces
)


In [20]:
# Remove words and clean up extra spaces
relevant_articles.loc[:, 'Full_text_min'] = (
    relevant_articles.loc[:, 'Full_text_min']
    .str.replace(pattern, '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with single space
    .str.strip()  # Remove leading/trailing spaces
)

In [21]:
# Format files (one word per column)
relevant_articles_format_title = format_file(relevant_articles, 'Title_min')
relevant_articles_format_abstract = format_file(relevant_articles, 'Abstract_min')

In [22]:
relevant_articles_format_full_text = format_file(relevant_articles.dropna(subset='Full_text_min'), 'Full_text_min')

In [24]:
relevant_articles_format_title = relevant_articles_format_title.map(
    lambda x: None if isinstance(x, str) and x in words_to_remove else x
)
relevant_articles_format_abstract = relevant_articles_format_abstract.map(
    lambda x: None if isinstance(x, str) and x in words_to_remove else x
)
relevant_articles_format_full_text = relevant_articles_format_full_text.map(
    lambda x: None if isinstance(x, str) and x in words_to_remove else x
)

In [26]:
relevant_articles_format_title.iloc[0:10,0:6]

,0,1,2,3,4,5
10618204,zeaxanthin,pigments,application,techniques,Synechocystis,sp
10618209,Cupriavidus,necator,flavohemoprotein,Vitreoscilla,hemoglobin,reductase
10631776,Environmental,biotechnology,NaN,NaN,NaN,NaN
10649237,Altered,pyruvate,kinase,phosphofructokinase,glycolytic,resting
10649449,Cloning,characterization,Yarrowia,lipolytica,squalene,SQS1
10653745,poly,hydroxyalkanoic,Escherichia,coli,NaN,NaN
10662693,Formation,complexes,subunits,picromycin,erythromycin,oleandomycin
10689078,Cupriavidus,necator,transformation,cloned,phbCAB,investigation
10699845,process,kinetics,Reprinted,Journal,Biochemical,Microbiological
10708651,cytochrome,p450,cam,CYP101,oxidation,polycyclic


<a class="anchor" id="Extract_organism_5"></a>
# <span style="color: #2a9466">Extract organism</span>

<a class="anchor" id="From_title_6"></a>
## <span style="color: #55aa66">From title</span>

In [27]:
relevant_articles_format_title.head()

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
10618204,zeaxanthin,pigments,application,techniques,Synechocystis,sp,PCC,6803,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10618209,Cupriavidus,necator,flavohemoprotein,Vitreoscilla,hemoglobin,reductase,fusion,hypoxic,Escherichia,coli,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10631776,Environmental,biotechnology,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10649237,Altered,pyruvate,kinase,phosphofructokinase,glycolytic,resting,Escherichia,coli,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10649449,Cloning,characterization,Yarrowia,lipolytica,squalene,SQS1,complementation,Saccharomyces,cerevisiae,erg9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
relevant_articles.loc[:, 'Organism'] = detect_organisms_vectorized(
    relevant_articles_format_title,
)


In [29]:
print(len(relevant_articles['Organism'].dropna()))
title_yeast_organism_count = len(relevant_articles['Organism'].dropna())

# remove general yeast organism
yeast_data = relevant_articles[
    relevant_articles['Organism'].apply(lambda orgs: isinstance(orgs, str) and orgs== 'yeast')
].copy()

relevant_articles.loc[relevant_articles.index.isin(yeast_data.index),'Organism']=np.nan

title_organism_count = len(relevant_articles['Organism'].dropna())
from_title = relevant_articles.Organism.dropna().index

10994


In [30]:
print(f"Found organisms in {title_organism_count} articles from titles, ({title_yeast_organism_count} taking into account 'yeast' as organism)")


Found organisms in 10383 articles from titles, (10994 taking into account 'yeast' as organism)


<a class="anchor" id="From_abstract_7"></a>
## <span style="color: #55aa66">From abstract</span>

In [31]:
# Get from abstract, as in 10618209 the gene from the organism if from the organism mentioned in the abstract, not the title 
relevant_articles.loc[:, 'Extra_organism_Abstract'] = detect_organisms_vectorized(
    relevant_articles_format_abstract,
)


In [32]:
articles_no_organism_from_title = relevant_articles.loc[relevant_articles.Organism.isna()].copy()
abstract_search = relevant_articles_format_abstract.loc[articles_no_organism_from_title.index,:]
articles_no_organism_from_title['Organism'] = detect_organisms_vectorized(
    abstract_search,
)

In [33]:
#articles_no_organism_from_title = relevant_articles.loc[relevant_articles.Organism.isna()].copy()
#articles_no_organism_from_title['Organism'] = relevant_articles_format_abstract.loc[articles_no_organism_from_title.index,:].apply(find_full_organism_by_genus,axis=1)
print(len(articles_no_organism_from_title['Organism'].dropna()))
abstract_yeast_organism_count = len(articles_no_organism_from_title['Organism'].dropna())

#remove general organism 'yeast'
yeast_data_abstract = relevant_articles[
    relevant_articles['Organism'].apply(lambda orgs: isinstance(orgs, str) and orgs== 'yeast')
].copy()

articles_no_organism_from_title.loc[articles_no_organism_from_title.index.isin(yeast_data_abstract.index),'Organism']=np.nan
relevant_articles.loc[relevant_articles.index.isin(yeast_data_abstract.index),'Organism']=np.nan

from_abstract = articles_no_organism_from_title.Organism.dropna().index



relevant_articles.loc[from_abstract,'Organism'] = articles_no_organism_from_title['Organism']

abstract_organism_count = len(articles_no_organism_from_title['Organism'].dropna())


3343


In [34]:
print(f"Found organisms in {abstract_organism_count} articles from abstracts,  ({abstract_yeast_organism_count} taking into account 'yeast' as organism)")

Found organisms in 3343 articles from abstracts,  (3343 taking into account 'yeast' as organism)


<a class="anchor" id="From_full_text_7"></a>
## <span style="color: #55aa66">From full text</span>

In [35]:
articles_still_missing_products = relevant_articles.loc[relevant_articles.Organism.isna()].copy()
#still missing articles with full text
relevant_articles_format_full_text.loc[relevant_articles_format_full_text.index.isin(articles_still_missing_products.index),:]


,0,1,2,3,4,5,6,7,8,9,...,33689,33690,33691,33692,33693,33694,33695,33696,33697,33698
10662693,cm7204,qxd,03,03,2000,08,38,Page,77,Formation,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11033080,Aminoacyl,SNACs,small,molecule,condensation,domains,nonribosomal,peptide,David,E,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11150509,FEBS,24432,Letters,487,2000,199,202,Intracellular,trehalose,osmotolerance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11251290,Chemistry,Biology,8,2001,157,178,www,elsevier,com,locate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11294879,JOURNAL,BIOLOGICAL,CHEMISTRY,Vol,276,24,Issue,June,15,21500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40426207,Introduction,CRISPR,Cas9,based,edit,regions,pikromycin,cluster,S,venezuelae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40430181,Introduction,2021,research,group,led,Kazak,Spiegelman,cycle,creatine,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40468204,Introduction,untargeted,high,throughput,sequencing,technologies,investigate,differences,tissues,Additionally,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40469736,Introduction,RNA,interference,RNAi,technology,downregulate,uridine,diphosphate,pyrophosphorylase,UGPase,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
articles_still_missing_products = relevant_articles.loc[relevant_articles.Organism.isna()].copy()
#still missing articles with full text
articles_to_get = relevant_articles_format_full_text.loc[relevant_articles_format_full_text.index.isin(articles_still_missing_products.index),:]
articles_still_missing_products['Organism'] = detect_organisms_vectorized(
    articles_to_get,
)

In [37]:
#articles_still_missing_products = relevant_articles.loc[relevant_articles.Organism.isna()].copy()
#still missing articles with full text
#articles_to_get = relevant_articles_format_full_text.loc[relevant_articles_format_full_text.index.isin(articles_still_missing_products.index),:]

#articles_still_missing_products['Organism'] = articles_to_get.apply(find_full_organism_by_genus,axis=1)
print(len(articles_still_missing_products['Organism'].dropna()))
full_text_yeast_organism_count = len(articles_still_missing_products['Organism'].dropna())

#remove general organism 'yeast'
yeast_data_full_text = relevant_articles[
    relevant_articles['Organism'].apply(lambda orgs: isinstance(orgs, str) and orgs== 'yeast')
].copy()

articles_still_missing_products.loc[articles_still_missing_products.index.isin(yeast_data_full_text.index),'Organism']=np.nan
relevant_articles.loc[relevant_articles.index.isin(yeast_data_full_text.index),'Organism']=np.nan

from_full_text = articles_still_missing_products.Organism.dropna().index


relevant_articles.loc[from_full_text,'Organism'] = articles_still_missing_products['Organism']

full_text_organism_count = len(articles_still_missing_products['Organism'].dropna())


1136


In [38]:
print(f"Found organisms in {full_text_organism_count} articles from full text")

Found organisms in 1136 articles from full text


In [39]:
relevant_articles.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Abstract_min,Title_min,Full_text_min,Organism,Extra_organism_Abstract
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,psbAII locus integration platform overexpress ...,zeaxanthin pigments application techniques Syn...,NaN,[Synechocystis sp PCC 6803],[Synechocystis sp PCC 6803]
10618209,Expression of Cupriavidus necator flavohemopro...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",<NA>,vhb hemoglobin Vitreoscilla sp. (VHb) organism...,Cupriavidus necator flavohemoprotein Vitreosci...,NaN,"[Cupriavidus necator, Escherichia coli]","[Vitreoscilla sp, Cupriavidus necator, Escheri..."
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,<NA>,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",<NA>,interest biotechnology need feed world's popul...,Environmental biotechnology.,NaN,None,None
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",<NA>,Glycolytic resting Escherichia coli overexpres...,Altered pyruvate kinase co-overexpression phos...,NaN,[Escherichia coli],"[Escherichia coli, Zymomonas mobilis]"
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,<NA>,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",<NA>,"squalene (SQS) encodes , farnesyl-diphosphate ...",Cloning characterization Yarrowia lipolytica s...,NaN,"[Yarrowia lipolytica, Saccharomyces cerevisiae]","[Yarrowia lipolytica, Saccharomyces cerevisiae]"


In [40]:
relevant_articles.Organism.dropna()

10618204                        [Synechocystis sp PCC 6803]
10618209            [Cupriavidus necator, Escherichia coli]
10649237                                 [Escherichia coli]
10649449    [Yarrowia lipolytica, Saccharomyces cerevisiae]
10653745                                 [Escherichia coli]
                                 ...                       
40572067                           [Bacillus licheniformis]
40572208        [Paenibacillus polymyxa, Bacillus subtilis]
40577193                         [Acetivibrio thermocellus]
40578703                         [Saccharomyces cerevisiae]
40579636                       [Bacillus paralicheniformis]
Name: Organism, Length: 14527, dtype: object

In [41]:
# NEW: Add origin tracking column
relevant_articles['Organism_Source'] = 'not_found'
relevant_articles.loc[from_title, 'Organism_Source'] = 'title'
relevant_articles.loc[from_abstract, 'Organism_Source'] = 'abstract'
relevant_articles.loc[from_full_text, 'Organism_Source'] = 'full_text'

relevant_articles['Organism_Source'] = relevant_articles['Organism_Source'].astype('category')


In [42]:
relevant_articles.Organism_Source.value_counts()

Organism_Source
title        10383
abstract      3343
not_found     1406
full_text     1136
Name: count, dtype: int64

<a class="anchor" id="Change_names_8"></a>
# <span style="color: #2a9466">Change names </span>

Change names like E. coli to Escherichia coli 

In [43]:
organisms = list(relevant_articles.Organism.dropna().values)
organisms = [item for sublist in organisms for item in sublist if len(item)>1]

In [44]:
organisms[0:10]

['Synechocystis sp PCC 6803',
 'Cupriavidus necator',
 'Escherichia coli',
 'Escherichia coli',
 'Yarrowia lipolytica',
 'Saccharomyces cerevisiae',
 'Escherichia coli',
 'Streptomyces lividans',
 'Saccharopolyspora erythraea',
 'Streptomyces antibioticus']

In [45]:
name_dictionary = make_name_dictionary(organisms)

In [46]:
mask = ~relevant_articles.Organism.isna()
relevant_articles.loc[mask, 'Full_name'] = relevant_articles.loc[mask, 'Organism'].apply(
    lambda x: get_full_name(x, name_dictionary=name_dictionary) if isinstance(x, list) else None
)

In [47]:
relevant_articles.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Abstract_min,Title_min,Full_text_min,Organism,Extra_organism_Abstract,Organism_Source,Full_name
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,psbAII locus integration platform overexpress ...,zeaxanthin pigments application techniques Syn...,NaN,[Synechocystis sp PCC 6803],[Synechocystis sp PCC 6803],title,[Synechocystis sp PCC 6803]
10618209,Expression of Cupriavidus necator flavohemopro...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",<NA>,vhb hemoglobin Vitreoscilla sp. (VHb) organism...,Cupriavidus necator flavohemoprotein Vitreosci...,NaN,"[Cupriavidus necator, Escherichia coli]","[Vitreoscilla sp, Cupriavidus necator, Escheri...",title,"[Cupriavidus necator, Escherichia coli]"
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,<NA>,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",<NA>,interest biotechnology need feed world's popul...,Environmental biotechnology.,NaN,None,None,not_found,NaN
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",<NA>,Glycolytic resting Escherichia coli overexpres...,Altered pyruvate kinase co-overexpression phos...,NaN,[Escherichia coli],"[Escherichia coli, Zymomonas mobilis]",title,[Escherichia coli]
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,<NA>,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",<NA>,"squalene (SQS) encodes , farnesyl-diphosphate ...",Cloning characterization Yarrowia lipolytica s...,NaN,"[Yarrowia lipolytica, Saccharomyces cerevisiae]","[Yarrowia lipolytica, Saccharomyces cerevisiae]",title,"[Yarrowia lipolytica, Saccharomyces cerevisiae]"


filter_top_30_org = filter_top_30.Title.str.replace('\xa0',' ' )
filter_top_30_org = filter_top_30_org.str.replace('.','',regex=False).str.split(' ')
filter_top_30.loc[:,'Organism'] = filter_top_30_org.apply(checar_lista)

In [48]:
bool_incons = (~relevant_articles.loc[:,'Organism'].isna()) & (relevant_articles.loc[:,'Full_name'].isna())

In [49]:
len(relevant_articles.loc[:,'Organism'].dropna())

14527

In [50]:
relevant_articles.loc[bool_incons,['Organism','Full_name']]

,Organism,Full_name
11494221,yeast,None
12620118,yeast,None
17439666,yeast,None
18707057,yeast,None
19038030,yeast,None
...,...,...
38961423,yeast,None
39416448,yeast,None
39994647,yeast,None
40061181,yeast,None


In [51]:
len(relevant_articles.loc[:,'Full_name'].dropna())

14445


19000

<a class="anchor" id="Result_9"></a>
# <span style="color: #2a9466">Result</span>

In [52]:
relevant_articles.loc[:,'Full_name'].explode().value_counts().head(20)

Full_name
Escherichia coli              4304
Saccharomyces cerevisiae      2297
Corynebacterium glutamicum     677
Yarrowia lipolytica            512
Bacillus subtilis              508
Pseudomonas putida             443
Komagataella pastoris          383
Cupriavidus necator            207
Synechocystis sp PCC 6803      173
Synechococcus elongatus        170
Agrobacterium tumefaciens      166
Lactococcus lactis             152
Aspergillus niger              149
Zymomonas mobilis              134
Clostridium acetobutylicum     131
Trichoderma reesei             130
Chlamydomonas reinhardtii      124
Streptomyces coelicolor        110
Acetivibrio thermocellus        94
Klebsiella pneumoniae           90
Name: count, dtype: int64

In [53]:
relevant_articles.Organism.apply(remove_duplicated_organisms) 

10618204                        [Synechocystis sp PCC 6803]
10618209            [Escherichia coli, Cupriavidus necator]
10631776                                               None
10649237                                 [Escherichia coli]
10649449    [Saccharomyces cerevisiae, Yarrowia lipolytica]
                                 ...                       
40572208        [Paenibacillus polymyxa, Bacillus subtilis]
40573728                                               None
40577193                         [Acetivibrio thermocellus]
40578703                         [Saccharomyces cerevisiae]
40579636                       [Bacillus paralicheniformis]
Name: Organism, Length: 16268, dtype: object

In [54]:
relevant_articles.Organism = relevant_articles.Organism.apply(remove_duplicated_organisms) 
relevant_articles.Full_name = relevant_articles.Full_name.apply(remove_duplicated_organisms) 

In [ ]:
relevant_articles.Full_name.explode().value_counts().head(15)

In [49]:
check_save_file(relevant_articles, output_file, 'Articles')

Saved file in: /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Output/Articles/full_text_classified_w_org_25_09_30.json
